In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset


In [2]:
class PreprocessingBase:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        if folder_path is None:
            raise ValueError("folder_path cannot be None")

    def common_preprocessing(self):
        raise NotImplementedError("Must implement common_preprocessing method.")


In [3]:
import os
import pickle
from PIL import Image
import torch

def check_path_exists(folder_path, filename):
    path = os.path.join(folder_path, filename)
    if not os.path.exists(path):
        raise ValueError(f"{path} does not exist")
    return True


def get_sub_folders_names(folder_path):
    sub_folders_name = [
        class_name
        for class_name in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, class_name))
    ]
    return sub_folders_name


def save_pickle(folder_path, obj, filename):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(folder_path, filename):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, "rb") as f:
        obj = pickle.load(f)
    return obj


def lower_data(df):
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(lambda x: x.lower() if isinstance(x, str) else x)


def drop_columns(X, col_drop):
    col_drop = list(col_drop.keys())
    if col_drop:
        X.drop(columns=col_drop, inplace=True, errors="ignore")


def is_valid_image(image_path):
    valid_extension = (".jpg", ".png", ".jpeg")

    if not os.path.isfile(image_path):
        return False
    if image_path.endswith(valid_extension):
        try:
            with Image.open(image_path) as img:
                img.verify()
            return True
        except Exception:
            return False
    return False
def collect_function(batch):
    images=[item[0]for item in batch]
    labels=[item[1]for item in batch]
    images_stack=torch.stack(images,dim=0)
    if labels[0] is None:
        return images_stack
    labels=torch.tensor(labels,dtype=torch.long)
    return images_stack,labels

In [4]:
import os
import pandas as pd


class ImageTrainingPreprocessing(PreprocessingBase):
    def __init__(
        self,
        training_folder_images,
        folder_path,
        validation_folder_images,
        split_training,
        val_size,
        random_state,
        images_size,
        augment=True,
        batch_size=16,
        augmentation_probability=0.5,
        horizontal_flip=True,
        vertical_flip=True,
        rotation=True,
        rotation_angle=30,
        brightness=True,
        brightness_factors=(0.8, 1.2),
        contrast=True,
        contrast_factors=(0.8, 1.2),
    ):
        super().__init__(folder_path)
        self.training_folder_images = training_folder_images
        self.validation_folder_images = validation_folder_images
        self.split_training = split_training
        self.val_size = val_size
        self.random_state = random_state
        self.image_size = images_size
        self.augment = augment
        self.batch_size = batch_size
        self.horizontal_flip = horizontal_flip
        self.vertical_flip = vertical_flip
        self.augmentation_probability = augmentation_probability
        self.rotation = rotation
        self.rotation_angle = rotation_angle
        self.brightness = brightness
        self.brightness_factors = brightness_factors
        self.contrast = contrast
        self.contrast_factors = contrast_factors
        self.report_data = {}
        if self.training_folder_images is None:
            raise ValueError("Training folder must be not null")
        check_path_exists(self.training_folder_images, "")
        if self.validation_folder_images is not None:
            check_path_exists(self.validation_folder_images, "")
            if self.split_training == True:
                raise ValueError(
                    "The validation folder not none so there is no need for splitting data again to training and validation"
                )
            if self.training_folder_images == self.validation_folder_images:
                raise ValueError(
                    "The validation folder and training folder must be different"
                )

        if (not (isinstance(self.val_size, (int, float)))) or (
            not (0 < self.val_size < 0.5)
        ):
            raise ValueError("Test size is invalid it must be less than 0.5")
        if (self.random_state is not None) and not (isinstance(self.random_state, int)):
            raise ValueError("Random state is invalid it must be integer or None")
        if not isinstance(self.image_size, tuple):
            raise ValueError("Image size must be tuple")
        if not isinstance(self.image_size[0], int) or not isinstance(
            self.image_size[1], int
        ):
            raise ValueError("Image size is not integer")
        if not (32 <= self.image_size[0] <= 1024) or not (
            32 <= self.image_size[1] <= 1024
        ):
            raise ValueError(
                "Image size is must be greater than or equal 32 and less than or equal 1024"
            )
        if not isinstance(self.image_size, tuple):
            raise ValueError("Image size must be tuple")
        if not isinstance(self.image_size[0], int) or not isinstance(
            self.image_size[1], int
        ):
            raise ValueError("Image size is not integer")
        if not (32 <= self.image_size[0] <= 1024) or not (
            32 <= self.image_size[1] <= 1024
        ):
            raise ValueError(
                "Image size is must be greater than or equal 32 and less than or equal 1024"
            )

        if not isinstance(self.augment, bool):
            raise ValueError("augment must be a boolean")

        if not isinstance(self.horizontal_flip, bool):
            raise ValueError("horizontal_flip must be a boolean")

        if not isinstance(self.vertical_flip, bool):
            raise ValueError("vertical_flip must be a boolean")

        if not isinstance(self.rotation, bool):
            raise ValueError("rotation must be a boolean")

        if not isinstance(self.brightness, bool):
            raise ValueError("brightness must be a boolean")

        if not isinstance(self.contrast, bool):
            raise ValueError("contrast must be a boolean")
        if not isinstance(self.augmentation_probability, (int, float)) or not (
            0 <= self.augmentation_probability <= 1
        ):
            raise ValueError("augmentation_probability must be between [0,1]")
        if not isinstance(self.rotation_angle, (int, float)) or self.rotation_angle < 0:
            raise ValueError("rotation_angle mustn't be negative")

        if (
            not isinstance(self.brightness_factors, tuple)
            or len(self.brightness_factors) != 2
            or not all(isinstance(x, (int, float)) for x in self.brightness_factors)
        ):
            raise ValueError(
                "brightness_factors must be tuple of two items float or integers"
            )
        if (
            not isinstance(self.contrast_factors, tuple)
            or len(self.contrast_factors) != 2
            or not all(isinstance(x, (int, float)) for x in self.contrast_factors)
        ):
            raise ValueError(
                "contrast_factors must be tuple of two items float or integers"
            )
        os.makedirs(self.folder_path, exist_ok=True)

    def get_sample_random(self, data_type, images_path, labels, num_samples=5):
        df = (
            pd.DataFrame(
                {
                    f"{data_type}_path": images_path,
                    f"{data_type}_labels": labels,
                }
            )
            .sample(frac=1, random_state=self.random_state)
            .reset_index(drop=True)
        ).head(num_samples)
        return df


In [5]:
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageEnhance
import numpy as np
import random


class ClassificationImageDataset(Dataset):
    def __init__(
        self,
        image_paths,
        labels=None,
        image_size=(224, 224),
        augment=True,
        augmentation_probability=0.5,
        horizontal_flip=True,
        vertical_flip=True,
        rotation=True,
        rotation_angle=30,
        brightness=True,
        brightness_factors=(0.8, 1.2),
        contrast=True,
        contrast_factors=(0.8, 1.2),
    ):
        self.image_paths = image_paths
        self.labels = labels
        self.image_size = image_size
        self.augment = augment
        self.horizontal_flip = horizontal_flip
        self.vertical_flip = vertical_flip
        self.augmentation_probability = augmentation_probability
        self.rotation = rotation
        self.rotation_angle = rotation_angle
        self.brightness = brightness
        self.brightness_factors = brightness_factors
        self.contrast = contrast
        self.contrast_factors = contrast_factors

    def __len__(self):
        return len(self.image_paths)

    def load_image(self, index):
        path = self.image_paths[index]
        img = Image.open(path).convert("RGB")
        img = img.resize(self.image_size)
        if self.augment:
            img = self.augmentation(img)
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_transpose = np.transpose(img_array, (2, 0, 1))
        img_tensor = torch.tensor(img_transpose, dtype=torch.float32)
        return img_tensor

    def random_horizontal_flip(self, img):
        if self.horizontal_flip and random.random() < self.augmentation_probability:
            return img.transpose(Image.FLIP_LEFT_RIGHT)
        return img

    def random_vertical_flip(self, img):
        if self.vertical_flip and random.random() < self.augmentation_probability:
            return img.transpose(Image.FLIP_TOP_BOTTOM)
        return img

    def random_rotation(self, img):
        if self.rotation and random.random() < self.augmentation_probability:
            random_angle = random.uniform(-1 * self.rotation_angle, self.rotation_angle)
            return img.rotate(random_angle)
        return img

    def random_brightness(self, img):
        if self.brightness and random.random() < self.augmentation_probability:
            random_factor = random.uniform(*self.brightness_factors)
            return ImageEnhance.Brightness(img).enhance(random_factor)
        return img

    def random_contrast(self, img):
        if self.contrast and random.random() < self.augmentation_probability:
            random_factor = random.uniform(*self.contrast_factors)
            return ImageEnhance.Contrast(img).enhance(random_factor)
        return img

    def augmentation(self, img):
        img = self.random_horizontal_flip(img)
        img = self.random_vertical_flip(img)
        img = self.random_rotation(img)
        img = self.random_brightness(img)
        img = self.random_contrast(img)
        return img

    def __getitem__(self, index):
        img_tensor = self.load_image(index)

        label_tensor = None
        if self.labels is not None:
            label_tensor = torch.tensor(self.labels[index], dtype=torch.long)
        return img_tensor, label_tensor


In [6]:
import os

from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split


class ClassificationImageTrainingPreprocessing(ImageTrainingPreprocessing):
    def __init__(
        self,
        training_folder_images,
        folder_path,
        validation_folder_images=None,
        split_training=False,
        val_size=0.2,
        random_state=42,
        images_size=(224, 224),
        augment=True,
        batch_size=16,
        augmentation_probability=0.5,
        horizontal_flip=True,
        vertical_flip=True,
        rotation=True,
        rotation_angle=30,
        brightness=True,
        brightness_factors=(0.8, 1.2),
        contrast=True,
        contrast_factors=(0.8, 1.2),
    ):
        super().__init__(
            training_folder_images,
            folder_path,
            validation_folder_images,
            split_training,
            val_size,
            random_state,
            images_size,
            augment,
            batch_size,
            augmentation_probability,
            horizontal_flip,
            vertical_flip,
            rotation,
            rotation_angle,
            brightness,
            brightness_factors,
            contrast,
            contrast_factors
        )
        self.training_class = get_sub_folders_names(self.training_folder_images)
        self.validation_class = None
        self.training_folder_invalid_images = []
        self.training_folder_invalid_images_labels = []
        self.validation_folder_invalid_images = []
        self.validation_folder_invalid_images_labels = []
        if self.validation_folder_images is not None:
            self.validation_class = get_sub_folders_names(self.validation_folder_images)
            for class_name in self.validation_class:
                if class_name not in self.training_class:
                    raise ValueError(
                        f"This category {class_name} not in the training categories which are {self.training_class}"
                    )
        self.validation_folder_images_paths, self.validation_folder_images_labels = (
            None,
            None,
        )
        data_info = {"image_size": self.image_size}
        save_pickle(self.folder_path, data_info, "data_info.pkl")

    def get_classes_mapping(self):
        self.class2label_mapping = {}
        self.label2class_mapping = {}
        for idx, class_name in enumerate(sorted(self.training_class)):
            self.class2label_mapping[class_name] = idx
            self.label2class_mapping[idx] = class_name
        save_pickle(
            self.folder_path, self.class2label_mapping, "class2label_mapping.pkl"
        )
        save_pickle(
            self.folder_path, self.label2class_mapping, "label2class_mapping.pkl"
        )

    def data_preparing(
        self, folder_path, invalid_list_paths, invalid_list_labels, classes_name
    ):
        paths = []
        labels = []
        for class_name in classes_name:
            sub_folder_path = os.path.join(folder_path, class_name)
            for path in os.listdir(sub_folder_path):
                path = os.path.join(sub_folder_path, path)
                mapping = self.class2label_mapping[class_name]
                if is_valid_image(path):
                    paths.append(path)
                    labels.append(mapping)
                else:
                    invalid_list_paths.append(path)
                    invalid_list_labels.append(mapping)
        return paths, labels

    def data_overview(self):
        train_df = self.get_sample_random(
            "training_folder",
            self.training_folder_images_paths,
            self.training_folder_images_labels,
        )
        self.report_data["data_overview"] = {}
        self.report_data["data_overview"]["training_folder"] = {
            "classes": self.training_class,
            "images_len": len(self.training_folder_images_paths),
            "random_sample": train_df.head(),
            "invalid_len": len(self.training_folder_invalid_images),
            "invalid_images": self.training_folder_invalid_images,
            "mapping": self.class2label_mapping,
        }

        if self.validation_folder_images is not None:
            val_df = self.get_sample_random(
                "validation_folder",
                self.validation_folder_images_paths,
                self.validation_folder_images_labels,
            )
            self.report_data["data_overview"]["validation_folder"] = {
                "classes": self.validation_class,
                "images_len": len(self.validation_folder_images_paths),
                "random_sample": val_df.head(),
                "invalid_len": len(self.validation_folder_invalid_images),
                "invalid_images": self.validation_folder_invalid_images,
            }

    def splitting(self):
        if self.split_training:
            (
                self.training_paths,
                self.validation_paths,
                self.training_labels,
                self.validation_labels,
            ) = train_test_split(
                self.training_folder_images_paths,
                self.training_folder_images_labels,
                test_size=self.val_size,
                random_state=self.random_state,
            )
            self.report_data["split_data"] = {
                "validation_size": self.val_size,
                "training_data_size": len(self.training_paths),
                "random_training_sample": self.get_sample_random(
                    "training", self.training_paths, self.training_labels
                ),
                "validation_data_size": len(self.validation_paths),
                "random_validation_sample": self.get_sample_random(
                    "validation", self.validation_paths, self.validation_labels
                ),
            }
        else:
            (
                self.training_paths,
                self.validation_paths,
                self.training_labels,
                self.validation_labels,
            ) = (
                self.training_folder_images_paths,
                self.validation_folder_images_paths,
                self.training_folder_images_labels,
                self.validation_folder_images_labels,
            )



    def get_loaders(self):
        training_dataset = ClassificationImageDataset(
            self.training_paths,
            self.training_labels,
            self.image_size,
            self.augment,
            self.augmentation_probability,
            self.horizontal_flip,
            self.vertical_flip,
            self.rotation,
            self.rotation_angle,
            self.brightness,
            self.brightness_factors,
            self.contrast,
            self.contrast_factors,
        )
        self.training_loader = DataLoader(
            training_dataset, batch_size=self.batch_size, shuffle=True
        )
        self.validation_loader = None
        if self.validation_paths is not None:
            validation_dataset = ClassificationImageDataset(
                self.validation_paths,
                self.validation_labels,
                self.image_size,
                False,
                self.augmentation_probability,
                self.horizontal_flip,
                self.vertical_flip,
                self.rotation,
                self.rotation_angle,
                self.brightness,
                self.brightness_factors,
                self.contrast,
                self.contrast_factors,
            )
            self.validation_loader = DataLoader(
                validation_dataset, batch_size=self.batch_size, shuffle=False
            )

    def common_preprocessing(self):

        self.get_classes_mapping()
        self.training_folder_images_paths, self.training_folder_images_labels = (
            self.data_preparing(
                self.training_folder_images,
                self.training_folder_invalid_images,
                self.training_folder_invalid_images_labels,
                self.training_class,
            )
        )
        if self.validation_folder_images is not None:
            (
                self.validation_folder_images_paths,
                self.validation_folder_images_labels,
            ) = self.data_preparing(
                self.validation_folder_images,
                self.validation_folder_invalid_images,
                self.validation_folder_invalid_images_labels,
                self.validation_class,
            )
        self.data_overview()
        self.splitting()
        self.get_loaders()
      
        return self.training_loader, self.validation_loader


In [10]:

from torch.utils.data import DataLoader


class ClassificationImageTestingPreprocessing(PreprocessingBase):
    def __init__(
        self, image_paths, image_class_names=None, folder_path=None, batch_size=4
    ):
        super().__init__(folder_path)
        self.image_paths = image_paths
        self.image_class_names = image_class_names
        self.batch_size = batch_size
        self.valid_images = []
        self.valid_images_class_names = []
        self.invalid_images = []
        if not isinstance(self.image_paths, list):
            raise ValueError("Image paths must be list of paths")
        if self.image_class_names is not None and not isinstance(
            self.image_class_names, list
        ):
            raise ValueError("Class names must be list of class names")
        if self.image_class_names is not None and len(self.image_class_names) != len(
            self.image_paths
        ):
            raise ValueError("image paths length must equal class names length")
        for i, path in enumerate(self.image_paths):
            if is_valid_image(path):
                self.valid_images.append(path)
                if self.image_class_names is not None:
                    self.valid_images_class_names.append(self.image_class_names[i])
            else:
                self.invalid_images.append(path)

        check_path_exists(self.folder_path, "data_info.pkl")
        self.image_size = load_pickle(self.folder_path, "data_info.pkl")["image_size"]
        check_path_exists(self.folder_path, "class2label_mapping.pkl")
        self.class2label_mapping = load_pickle(
            self.folder_path, "class2label_mapping.pkl"
        )

    def common_preprocessing(self):
        labels = None
        if self.image_class_names is not None:
            labels = []
            for class_name in self.valid_images_class_names:
                if class_name not in self.class2label_mapping:
                    raise ValueError(f"this class name not in the training class")
                labels.append(self.class2label_mapping[class_name])
        dataset = ClassificationImageDataset(
            self.valid_images, labels, self.image_size, augment=False
        )
        testing_loader = DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=collect_function,
        )
        return testing_loader, self.invalid_images


In [7]:
training_preprocessor = ClassificationImageTrainingPreprocessing(
    training_folder_images="/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages",
    folder_path="/kaggle/working/PetImagesReport",
    validation_folder_images=None,
    split_training=True,
    val_size=0.2,
    random_state=23,
    images_size=(224, 224),
)
training_loader, validation_loader = training_preprocessor.common_preprocessing()

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
num_classes = len(training_preprocessor.class2label_mapping)

model = models.resnet18(weights="IMAGENET1K_V1")  
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10


for epoch in range(num_epochs):

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in training_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total


    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss/len(training_loader):.4f} | Train Acc: {train_accuracy:.2f}%")
    print(f"Val   Loss: {val_loss/len(validation_loader):.4f} | Val   Acc: {val_accuracy:.2f}%")
    print("-" * 50)





Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 186MB/s]
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch [1/10]
Train Loss: 0.3330 | Train Acc: 85.69%
Val   Loss: 0.2274 | Val   Acc: 90.46%
--------------------------------------------------
Epoch [2/10]
Train Loss: 0.2455 | Train Acc: 89.67%
Val   Loss: 0.2130 | Val   Acc: 90.08%
--------------------------------------------------
Epoch [3/10]
Train Loss: 0.2167 | Train Acc: 91.06%
Val   Loss: 0.1575 | Val   Acc: 93.24%
--------------------------------------------------
Epoch [4/10]
Train Loss: 0.1901 | Train Acc: 92.12%
Val   Loss: 0.2424 | Val   Acc: 89.00%
--------------------------------------------------
Epoch [5/10]
Train Loss: 0.1802 | Train Acc: 92.69%
Val   Loss: 0.1240 | Val   Acc: 94.96%
--------------------------------------------------
Epoch [6/10]
Train Loss: 0.1574 | Train Acc: 93.78%
Val   Loss: 0.1249 | Val   Acc: 94.96%
--------------------------------------------------
Epoch [7/10]
Train Loss: 0.1502 | Train Acc: 93.90%
Val   Loss: 0.1178 | Val   Acc: 95.54%
--------------------------------------------------
Epoch 

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  
        x = self.pool(F.relu(self.conv2(x)))  
        x = self.pool(F.relu(self.conv3(x))) 

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
num_classes = len(training_preprocessor.class2label_mapping)

model = SimpleCNN(num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10


for epoch in range(num_epochs):

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in training_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total


    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss/len(training_loader):.4f} | Train Acc: {train_accuracy:.2f}%")
    print(f"Val   Loss: {val_loss/len(validation_loader):.4f} | Val   Acc: {val_accuracy:.2f}%")
    print("-" * 50)





Using device: cuda


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch [1/10]
Train Loss: 0.6972 | Train Acc: 51.86%
Val   Loss: 0.6819 | Val   Acc: 55.84%
--------------------------------------------------
Epoch [2/10]
Train Loss: 0.6767 | Train Acc: 57.43%
Val   Loss: 0.6620 | Val   Acc: 61.18%
--------------------------------------------------
Epoch [3/10]
Train Loss: 0.6188 | Train Acc: 65.28%
Val   Loss: 0.5885 | Val   Acc: 68.14%
--------------------------------------------------
Epoch [4/10]
Train Loss: 0.5767 | Train Acc: 69.55%
Val   Loss: 0.5448 | Val   Acc: 71.82%
--------------------------------------------------
Epoch [5/10]
Train Loss: 0.5517 | Train Acc: 71.49%
Val   Loss: 0.5316 | Val   Acc: 73.04%
--------------------------------------------------
Epoch [6/10]
Train Loss: 0.5305 | Train Acc: 73.08%
Val   Loss: 0.5099 | Val   Acc: 75.34%
--------------------------------------------------
Epoch [7/10]
Train Loss: 0.5082 | Train Acc: 74.79%
Val   Loss: 0.4863 | Val   Acc: 77.24%
--------------------------------------------------
Epoch 

In [8]:
training_preprocessor = ClassificationImageTrainingPreprocessing(
    training_folder_images="/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages",
    folder_path="/kaggle/working/PetImagesReport",
    validation_folder_images=None,
    split_training=True,
    val_size=0.2,
    random_state=23,
    images_size=(224, 224),
    augment=False
)
training_loader, validation_loader = training_preprocessor.common_preprocessing()

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 28 * 28, 512) 
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  
        x = self.pool(F.relu(self.conv2(x)))  
        x = self.pool(F.relu(self.conv3(x)))  

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
num_classes = len(training_preprocessor.class2label_mapping)

model = SimpleCNN(num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10


for epoch in range(num_epochs):

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in training_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_accuracy = 100 * train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total


    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss/len(training_loader):.4f} | Train Acc: {train_accuracy:.2f}%")
    print(f"Val   Loss: {val_loss/len(validation_loader):.4f} | Val   Acc: {val_accuracy:.2f}%")
    print("-" * 50)





Using device: cuda


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch [1/10]
Train Loss: 0.6800 | Train Acc: 56.58%
Val   Loss: 0.6362 | Val   Acc: 62.92%
--------------------------------------------------
Epoch [2/10]
Train Loss: 0.6039 | Train Acc: 67.09%
Val   Loss: 0.5315 | Val   Acc: 73.30%
--------------------------------------------------
Epoch [3/10]
Train Loss: 0.4841 | Train Acc: 76.48%
Val   Loss: 0.5233 | Val   Acc: 74.62%
--------------------------------------------------
Epoch [4/10]
Train Loss: 0.3625 | Train Acc: 83.54%
Val   Loss: 0.5255 | Val   Acc: 77.38%
--------------------------------------------------
Epoch [5/10]
Train Loss: 0.1903 | Train Acc: 92.43%
Val   Loss: 0.7309 | Val   Acc: 76.00%
--------------------------------------------------
Epoch [6/10]
Train Loss: 0.0730 | Train Acc: 97.52%
Val   Loss: 1.0722 | Val   Acc: 75.80%
--------------------------------------------------
Epoch [7/10]
Train Loss: 0.0414 | Train Acc: 98.67%
Val   Loss: 1.3540 | Val   Acc: 74.50%
--------------------------------------------------
Epoch 

In [11]:
testing_loader, invalid_path = ClassificationImageTestingPreprocessing(
    ["/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages/Cat/10022.jpg", "/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages/Dog/10000.jpg"],
    image_class_names=["Cat", "Dog"],
    folder_path="/kaggle/working/PetImagesReport",
).common_preprocessing()

In [ ]:
model.eval()
with torch.no_grad():
        for images, labels in testing_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            
            _, predicted = torch.max(outputs, 1)
            print("predicted",predicted)
            print("corrected",labels)

predicted tensor([0, 1], device='cuda:0')
corrected tensor([0, 1], device='cuda:0')
